In [64]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


In [30]:
df = pd.read_csv(r"E:\ev battery\ev_battery_health.csv")

In [31]:
df.isnull().sum()

battery_id                            0
observation_id                        0
chemistry                          3459
manufacturer                       2461
vehicle_model                      3515
pack_capacity_kwh                  2302
manufacture_date                   1910
observation_date                   2216
battery_age_days                   3002
cycle_count                        2269
climate_zone                       2719
avg_daily_distance_km              2118
avg_charging_power_kw              2140
fast_charge_ratio                  3405
avg_depth_of_discharge_pct         2832
avg_state_of_charge_pct            3138
charge_cycles_per_month            2054
total_energy_throughput_kwh        3370
avg_ambient_temp_c                 3282
humidity_pct                       2737
altitude_m                         2322
internal_resistance_mohm           2104
voltage_avg_v                      2807
current_avg_a                      2487
charge_time_avg_min                2091


In [32]:
df.sample(5)

,battery_id,observation_id,chemistry,manufacturer,vehicle_model,pack_capacity_kwh,manufacture_date,observation_date,battery_age_days,cycle_count,climate_zone,avg_daily_distance_km,avg_charging_power_kw,fast_charge_ratio,avg_depth_of_discharge_pct,avg_state_of_charge_pct,charge_cycles_per_month,total_energy_throughput_kwh,avg_ambient_temp_c,humidity_pct,altitude_m,internal_resistance_mohm,voltage_avg_v,current_avg_a,charge_time_avg_min,peak_temp_during_fast_charge_c,capacity_fade_pct,service_count,last_service_date,bms_firmware_version,recall_flag,state_of_health_pct,remaining_useful_life_cycles,failure_risk_label,degradation_category
36462,BATT_00465,OBS_008951,LFP,PowerGrid Batteries,Sport-E9,60.4,NaN,2025-11-18,1046.0,934.0,hot_arid,43.1,49.0,0.339,43.7,73.5,18.3,52570.6,22.6,58.9,42.0,52.76,352.3,143.2,33.6,32.9,7.46,2.0,2024-12-15,BMS_v3.5,0.0,92.54,2407,Low,Normal
8747,BATT_00401,OBS_007706,NMC,NexCell,Sport-E9,81.6,2026-05-16,2026-08-13,89.0,109.0,tropical_humid,37.1,27.9,0.274,56.3,66.7,25.3,8041.0,23.8,76.4,112.0,NaN,363.6,148.9,96.3,29.8,3.00,1.0,2026-05-30,BMS_v3.5,0.0,97.00,1069,Low,Normal
5680,BATT_02084,OBS_040066,LMO,PowerGrid Batteries,Pickup-E7,54.1,2025-08-03,2026-02-16,197.0,160.0,tropical_humid,22.2,11.2,0.073,83.1,51.6,19.9,NaN,23.2,58.8,245.0,41.00,355.7,121.8,258.4,20.4,3.92,0.0,NaN,BMS_v3.5,0.0,96.08,721,Low,Normal
11407,BATT_00814,OBS_015598,LFP,NaN,Van-E4,67.4,2024-06-25,2024-12-22,180.0,229.0,hot_arid,24.4,10.7,0.046,54.0,76.3,30.1,15083.5,34.6,64.7,323.0,31.97,367.3,128.8,223.8,35.0,1.64,0.0,NaN,BMS_v3.0,NaN,98.36,3187,Low,Normal
37810,BATT_01466,OBS_028181,LFP,PowerGrid Batteries,Van-E4,76.5,2025-02-14,2026-09-04,567.0,594.0,NaN,42.5,53.7,0.449,64.8,68.4,22.2,41254.2,0.0,62.7,586.0,40.25,386.0,123.6,51.7,9.7,3.74,3.0,2026-01-08,BMS_v3.5,0.0,96.26,3430,Low,Normal


In [33]:
df["chemistry"].value_counts()

chemistry
NMC    15984
LFP    13945
NCA     9727
LMO     6885
Name: count, dtype: int64

In [34]:
df["manufacturer"].duplicated().sum()

np.int64(49991)

In [35]:
pd.set_option('display.max_columns', None)
df.head()

,battery_id,observation_id,chemistry,manufacturer,vehicle_model,pack_capacity_kwh,manufacture_date,observation_date,battery_age_days,cycle_count,climate_zone,avg_daily_distance_km,avg_charging_power_kw,fast_charge_ratio,avg_depth_of_discharge_pct,avg_state_of_charge_pct,charge_cycles_per_month,total_energy_throughput_kwh,avg_ambient_temp_c,humidity_pct,altitude_m,internal_resistance_mohm,voltage_avg_v,current_avg_a,charge_time_avg_min,peak_temp_during_fast_charge_c,capacity_fade_pct,service_count,last_service_date,bms_firmware_version,recall_flag,state_of_health_pct,remaining_useful_life_cycles,failure_risk_label,degradation_category
0,BATT_01748,OBS_033554,LMO,StormCell,Crossover-X2,48.5,2022-11-23,2024-10-16,693.0,991.0,cold,120.3,NaN,0.345,65.7,66.6,29.5,41968.2,-4.3,60.2,390.0,92.70,NaN,146.4,40.1,-1.3,19.13,4.0,2023-12-25,BMS_v2.1,0.0,80.87,0,High,Normal
1,BATT_00491,OBS_009428,LFP,VoltaCore,Sedan-E1,61.5,2023-03-06,2024-12-27,662.0,1535.0,cold,93.4,74.7,0.569,74.2,51.2,47.4,86773.6,-7.6,34.5,1873.0,49.95,357.3,176.0,38.0,5.3,8.19,0.0,NaN,BMS_v3.2,0.0,91.81,1739,Low,Normal
2,BATT_00011,OBS_000200,LFP,IonFlux,Van-E4,73.1,2025-06-24,2026-02-10,NaN,407.0,cold,45.0,81.7,0.641,71.3,70.2,34.8,28829.7,12.3,46.5,249.0,37.90,375.4,156.8,41.7,21.3,2.52,0.0,NaN,NaN,0.0,97.48,3248,Low,Normal
3,BATT_00651,OBS_012448,NMC,Amperion,NaN,75.6,2022-08-13,2022-09-15,33.0,29.0,cold,42.4,76.0,0.596,33.1,79.2,17.2,NaN,14.0,66.9,3.0,NaN,361.7,150.3,18.2,20.8,0.82,0.0,NaN,BMS_v3.5,0.0,99.18,1375,Low,Normal
4,BATT_02054,OBS_039490,NCA,VoltaCore,Hatch-E3,92.7,2024-09-27,2026-03-01,520.0,324.0,hot_arid,23.4,36.3,0.550,57.3,70.4,16.9,26916.1,33.2,71.7,206.0,68.34,374.0,159.5,79.2,40.1,8.14,0.0,NaN,BMS_v2.1,0.0,91.86,715,Low,Normal


In [36]:
df["bms_firmware_version"] = df["bms_firmware_version"].str.extract(r'v(\d+\.\d+)').astype(float)

In [37]:
pd.set_option('display.max_columns', None)
df.sample(5)

,battery_id,observation_id,chemistry,manufacturer,vehicle_model,pack_capacity_kwh,manufacture_date,observation_date,battery_age_days,cycle_count,climate_zone,avg_daily_distance_km,avg_charging_power_kw,fast_charge_ratio,avg_depth_of_discharge_pct,avg_state_of_charge_pct,charge_cycles_per_month,total_energy_throughput_kwh,avg_ambient_temp_c,humidity_pct,altitude_m,internal_resistance_mohm,voltage_avg_v,current_avg_a,charge_time_avg_min,peak_temp_during_fast_charge_c,capacity_fade_pct,service_count,last_service_date,bms_firmware_version,recall_flag,state_of_health_pct,remaining_useful_life_cycles,failure_risk_label,degradation_category
36469,BATT_01395,OBS_026869,NCA,GigaWatt Labs,Pickup-E7,NaN,2021-05-31,2021-11-16,169.0,59.0,hot_arid,13.8,24.2,0.230,74.4,66.7,5.9,4540.6,NaN,81.3,361.0,29.76,371.4,146.8,152.9,41.0,1.58,0.0,NaN,3.2,0.0,98.42,1489,Low,Normal
29014,BATT_01305,OBS_025091,LMO,StormCell,Compact-E1,46.8,2023-02-19,2023-06-04,105.0,83.0,tropical_humid,18.0,67.4,0.485,NaN,76.8,20.1,3718.8,46.6,77.0,105.0,34.06,359.0,146.4,22.3,54.0,NaN,1.0,2023-03-19,2.1,0.0,98.36,680,Low,Normal
14310,BATT_01165,OBS_022350,NMC,NaN,Compact-E1,79.1,2024-04-13,2024-04-18,5.0,4.0,NaN,37.6,29.8,0.434,82.9,NaN,17.8,299.8,17.1,53.6,238.0,32.01,352.3,127.2,145.0,24.9,0.06,0.0,NaN,2.1,0.0,99.94,1749,Low,Normal
7526,BATT_01414,OBS_027230,LFP,Amperion,NaN,60.4,2023-05-14,2024-04-02,324.0,387.0,mixed,14.8,14.0,0.088,61.8,73.8,23.2,23180.3,17.6,61.7,78.0,42.45,350.4,128.8,169.3,20.3,2.55,0.0,NaN,2.3,0.0,97.45,3224,Low,Normal
45922,BATT_02023,OBS_038871,NCA,StormCell,Compact-E1,71.5,NaN,2025-11-12,183.0,149.0,hot_arid,59.3,27.8,0.359,64.6,74.0,12.4,9283.8,48.5,72.2,85.0,32.53,361.3,144.1,95.1,54.5,NaN,0.0,NaN,3.2,0.0,97.46,1150,Low,Normal


In [38]:
df.drop(columns=["battery_id","observation_id","vehicle_model"],inplace=True)

In [39]:
date_columns = ["manufacture_date","observation_date", "last_service_date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])
    df[col + "_Year"] = df[col].dt.year
    df[col + "_Month"] = df[col].dt.month
    df[col + "_Day"] = df[col].dt.day

df = df.drop(columns=date_columns)

In [40]:
num_columns = [
    "pack_capacity_kwh",
    "manufacture_date_Year",
    "manufacture_date_Month",
    "manufacture_date_Day",
    "observation_date_Year",
    "observation_date_Month",
    "observation_date_Day",
    "last_service_date_Year",
    "last_service_date_Month",
    "last_service_date_Day",
    "battery_age_days",
    "cycle_count",
    "avg_daily_distance_km",
    "avg_charging_power_kw",
    "fast_charge_ratio",
    "avg_depth_of_discharge_pct",
    "avg_state_of_charge_pct",
    "charge_cycles_per_month",
    "total_energy_throughput_kwh",
    "avg_ambient_temp_c",
    "humidity_pct",
    "altitude_m",
    "internal_resistance_mohm",
    "voltage_avg_v",
    "current_avg_a",
    "charge_time_avg_min",
    "peak_temp_during_fast_charge_c",
    "capacity_fade_pct",
    "service_count",
    "recall_flag",
    "state_of_health_pct",
    "remaining_useful_life_cycles"
]

In [41]:
cat_val = ["chemistry","manufacturer","climate_zone","degradation_category"]

In [42]:
df["failure_risk_label"].value_counts()

failure_risk_label
Low       42735
Medium     4409
High       2856
Name: count, dtype: int64

In [43]:
y = df["failure_risk_label"]
X = df.drop(["failure_risk_label"],axis=1)

In [44]:
le = LabelEncoder()

colors = df["failure_risk_label"]

encoded = le.fit_transform(colors)

In [45]:
num_pipline = Pipeline([
    ("imputer", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())
])

In [46]:
cat_pipline = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("Ohe", OneHotEncoder(
    handle_unknown='ignore',
    drop=None,
    sparse_output=True,
    min_frequency=5
))
])

In [47]:
preprocess = ColumnTransformer([
    ("num", num_pipline, num_columns),
    ("cat", cat_pipline, cat_val)
])

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [49]:
model = Pipeline([
    ("processing", preprocess),
    ("model", DecisionTreeClassifier())
])

In [50]:
model.fit(X_train,y_train)

Pipeline(steps=[('processing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['pack_capacity_kwh',
                                                   'manufacture_date_Year',
                                                   'manufacture_date_Month',
                                                   'manufacture_date_Day',
                                                   'observation_date_Year',
                                                   'observation_date_Month',
                                                   'observation_date_Day',
                                                   'last_service_date_Year',
                                                   'last_service_dat...
                                                   'charge_time_avg_min',
                                                   'peak_temp_during_fast_charge_c',
                                                   'capacity_fade_pct',
                                                   'service_count',
                                                   'recall_flag', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 min_frequency=5))]),
                                                  ['chemistry', 'manufacturer',
                                                   'climate_zone',
                                                   'degradation_category'])])),
                ('model', DecisionTreeClassifier())])

In [51]:
y_pred = model.predict(X_test)


In [52]:



y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted"))
print("Recall:", recall_score(y_test, y_pred, average="weighted"))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted"))



Accuracy: 0.948
Precision: 0.9488662433524363
Recall: 0.948
F1 Score: 0.948410237436034
